# Clase 035 — Matplotlib: anatomía

**Parte 0** · VanderPlas cap. 4 § 4.1.

> 🎯 Figure → Axes → Artist. API OO en vez de pyplot estilo MATLAB.

> ⏱️ ~60 min

## ⚙️ Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
print('matplotlib:', plt.matplotlib.__version__)

## 1️⃣ Jerarquía

```
Figure (canvas, contiene N axes)
 └── Axes (un gráfico — el rectángulo con sus ejes)
      ├── Line2D, Scatter, Bar, ...   (Artists — lo dibujado)
      ├── XAxis / YAxis                 (los ejes con sus ticks)
      ├── Legend
      └── Title
```

**Figure** = el canvas (ventana, archivo). **Axes** = un gráfico (puedes tener varios en una figura). **Artist** = todo lo demás (líneas, puntos, texto).

## 2️⃣ pyplot vs API OO

```python
# ❌ pyplot — state-based (estilo MATLAB)
plt.plot(x, y)
plt.title('Título')
plt.xlabel('x')
plt.savefig('out.png')

# ✅ OO — explícito, escalable
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(x, y)
ax.set_title('Título')
ax.set_xlabel('x')
fig.savefig('out.png')
```

La OO te obliga a nombrar el axes que estás manipulando — esto escala a 4 subplots sin confusión.

## 3️⃣ El patrón canónico

In [ ]:
x = np.linspace(0, 2*np.pi, 200)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(x, np.sin(x), label='sin(x)', linewidth=2)
ax.plot(x, np.cos(x), label='cos(x)', linewidth=2, linestyle='--')
ax.set_title('sin y cos')
ax.set_xlabel('x (radianes)')
ax.set_ylabel('valor')
ax.legend(loc='upper right')
ax.grid(alpha=0.3)
ax.axhline(0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

## 4️⃣ Guardar — raster vs vector

```python
fig.savefig('out.png', dpi=300)          # raster, fixed resolution
fig.savefig('out.svg')                   # vector, escala infinita
fig.savefig('out.pdf', bbox_inches='tight')   # vector, ideal para LaTeX
```

- **PNG**: para web, presentaciones. DPI ≥ 150 para que se vea decente.
- **SVG/PDF**: para informes editables o LaTeX. Tamaño pequeño si no hay scatter denso.
- **`bbox_inches='tight'`**: recorta márgenes vacíos.
- **`facecolor='white'`**: por default fondo transparente — explicítalo si lo quieres blanco.

## 5️⃣ Liberar memoria en loops

Cada `plt.subplots()` deja una Figure viva en memoria. En notebooks que generan muchas figuras, esto causa OOM.

In [ ]:
import gc

antes = len(plt.get_fignums())
for i in range(20):
    fig, ax = plt.subplots()
    ax.plot([1, 2, 3])
    plt.close(fig)   # ← libera
    gc.collect()

print(f'figuras abiertas: {len(plt.get_fignums())} (esperado: {antes})')

## 6️⃣ `rcParams` — defaults globales

```python
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['lines.linewidth'] = 2
plt.rcParams['font.size'] = 12
plt.rcParams['axes.grid'] = True
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
```

O usa stylesheets predefinidos (clase 037): `plt.style.use('seaborn-v0_8-whitegrid')`.

## ✅ Checklist

- [ ] Entiendo Figure → Axes → Artist
- [ ] Uso `fig, ax = plt.subplots()` en vez de pyplot directo
- [ ] Guardo en formato apropiado (PNG/SVG/PDF)
- [ ] Cierro figuras explícitamente en loops
- [ ] Sé configurar rcParams

## 📝 Homework

Ver `README.md`. sin/cos publicable, savefig 3 formatos, loop sin leak, rcParams demo.

## 📖 Definiciones y características

**Figure**

El canvas/lienzo completo (ventana, archivo PNG). Una Figure contiene N Axes.

**Axes**

Un gráfico individual con sus ejes X/Y, ticks, labels, leyenda. **No es plural** de axis — es la palabra técnica de matplotlib para 'el rectángulo donde se dibuja'.

**Artist**

Todo lo dibujado: lines (Line2D), puntos (PathCollection), texto, leyenda. Cada element es un Artist con propiedades modificables.

**API OO vs pyplot**

**OO**: `fig, ax = plt.subplots(); ax.plot(...)` — explícito, escalable. **pyplot**: `plt.plot(...)` — estilo MATLAB, estado global, menos predecible. Usa OO siempre.

**`rcParams`**

Dict global con defaults de matplotlib: `plt.rcParams['figure.figsize'] = (10, 5)`. Modificar afecta todos los plots subsiguientes hasta reset.

## ⚠️ Errores comunes

| Síntoma / mensaje | Causa y cómo arreglar |
|---|---|
| Mi notebook se llena de memoria al hacer muchos plots | Cada `plt.subplots()` deja Figure abierta. **Fix**: `plt.close(fig)` después de mostrar/guardar, o `plt.close('all')` periódicamente. |
| `savefig('out.png')` da PNG con fondo transparente raro | Default es transparent. **Fix**: `fig.savefig('out.png', facecolor='white')` o `dpi=300, bbox_inches='tight'`. |
| Labels cortados al guardar | Bounding box no incluye text fuera del axes. **Fix**: `bbox_inches='tight'` en savefig o `constrained_layout=True` en subplots. |
| `plt.plot(x, y)` no muestra nada en notebook | Falta `%matplotlib inline` (default suele estarlo). En scripts standalone: `plt.show()` al final. |
| Texto en español sale mal | Default font no tiene tildes/ñ. **Fix**: `plt.rcParams['font.family'] = 'DejaVu Sans'` u otra Unicode-friendly. |

## ❓ Preguntas frecuentes

**❓ ¿`fig, ax = plt.subplots()` o `fig = plt.figure(); ax = fig.add_subplot()`?**

`subplots()` es el shortcut más usado. Da figure + axes en una línea, devuelve grid si pasas (n, m). El segundo es más explícito y útil para layouts custom (GridSpec).

**❓ ¿PNG, SVG o PDF?**

**PNG** para web/presentaciones (raster, DPI fijo). **SVG** para documentos editables / publicación (vector, escala infinita). **PDF** para LaTeX (también vector).

**❓ ¿Cuándo `constrained_layout=True` vs `tight_layout()`?**

**`constrained_layout=True`** (al crear figure): más nuevo, más confiable, maneja colorbars y leyendas externas mejor. **`tight_layout()`** (después): legacy, falla con elementos no estándar.

**❓ ¿Por qué mi plot se ve diferente entre script y notebook?**

Backend distinto: notebook usa `inline`, script usa `Qt5Agg`/`Tk`. DPI y tamaño cambian. Para consistencia: `plt.rcParams['figure.dpi'] = 100` explícito.

**❓ ¿Está bien usar `plt.plot()` directo?**

Para 1 plot rápido en notebook, OK. Para cualquier cosa más compleja (subplots, savefig, reutilizable), siempre API OO.

## 🔗 Referencias

- VanderPlas cap. 4 § 4.1
- [matplotlib quick start](https://matplotlib.org/stable/users/explain/quick_start.html)

➡️ **Siguiente:** [036 — line/scatter/bar/hist/box](../036-matplotlib-line-scatter-bar-histogram-boxplot/README.md)

## ✅ Soluciones de los ejercicios

Soluciones trabajadas y **ejecutables sin internet** de los ejercicios del README. Cada bloque incluye comentarios y comprobaciones (`assert`/`print`). Intenta resolverlos tú antes de mirar.

**Ejercicio 1 — Hello world.** Figura 8×4 con `y = sin(x)` en `[0, 2π]`, título y etiquetas.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
x = np.linspace(0, 2*np.pi, 200)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(x, np.sin(x))
ax.set_title('y = sin(x)'); ax.set_xlabel('x (rad)'); ax.set_ylabel('sin(x)')
# comprobamos el tamaño real de la figura
assert fig.get_size_inches().tolist() == [8.0, 4.0]
print('Figura', tuple(fig.get_size_inches()), '| título:', ax.get_title())
plt.close(fig)

**Ejercicio 2 — Dos líneas + leyenda.** `sin(x)` y `cos(x)` con colores distintos.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(x, np.sin(x), color='tab:blue',   label='sin(x)')
ax.plot(x, np.cos(x), color='tab:orange', label='cos(x)')
ax.legend()
assert len(ax.lines) == 2 and ax.get_legend() is not None
print('Líneas:', len(ax.lines),
      '| leyenda:', [t.get_text() for t in ax.get_legend().get_texts()])
plt.close(fig)

**Ejercicio 3 — Guarda 3 formatos.** PNG 100 DPI, PNG 300 DPI y SVG; compara tamaños.

In [ ]:
import os, tempfile
fig, ax = plt.subplots(figsize=(8, 4)); ax.plot(x, np.sin(x))
d = tempfile.mkdtemp(); sizes = {}
for name, kw in {'png_100': dict(format='png', dpi=100),
                 'png_300': dict(format='png', dpi=300),
                 'svg':     dict(format='svg')}.items():
    p = os.path.join(d, name); fig.savefig(p, **kw); sizes[name] = os.path.getsize(p)
plt.close(fig)
for k, v in sizes.items():
    print(f'{k:8s}: {v:>8,} bytes')
assert sizes['png_300'] > sizes['png_100']   # más DPI -> más píxeles -> más bytes
print('OK: 300 DPI pesa más que 100 DPI')

**Ejercicio 4 — Loop sin *leak*.** 20 figuras cerrando cada una con `plt.close(fig)`.

In [ ]:
for i in range(20):
    fig, ax = plt.subplots()
    ax.plot(np.random.rand(10))
    plt.close(fig)                 # <- clave: liberar memoria de cada figura
n = len(plt.get_fignums())
assert n == 0, f'Quedaron {n} figuras abiertas'
print('Figuras abiertas tras el loop:', n, '(sin leak)')

**Ejercicio 5 — rcParams.** Cambia `font.size` y `lines.linewidth` y verifica el efecto.

In [ ]:
import matplotlib as mpl
orig = (plt.rcParams['font.size'], plt.rcParams['lines.linewidth'])
with mpl.rc_context({'font.size': 16, 'lines.linewidth': 3.0}):
    assert plt.rcParams['font.size'] == 16
    assert plt.rcParams['lines.linewidth'] == 3.0
    print('Dentro del context -> font.size=16, linewidth=3.0')
# fuera del context, todo vuelve a los valores originales
assert (plt.rcParams['font.size'], plt.rcParams['lines.linewidth']) == orig
print('Fuera del context, restaurado a', orig)